# 05 — Model Evaluation

This notebook evaluates all trained models and performs error analysis.

**Objectives:**
- Compute classification metrics for all models
- Generate confusion matrices
- Compare models visually
- Analyze false positives and false negatives
- Discuss biases and limitations

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import json, joblib

from src.data_loader import load_and_merge_data, remove_duplicates
from src.preprocessing import clean_text, combine_title_text
from src.train import prepare_train_test_split
from src.evaluate import (
    evaluate_all_models, create_comparison_table,
    plot_confusion_matrix, plot_model_comparison,
    analyze_errors, save_all_metrics
)
from src.utils import get_model_dir

sns.set_style("whitegrid")
print("Libraries loaded.")

## 1. Prepare Data (Same Pipeline as Training)

In [ ]:
df = load_and_merge_data()
df, _ = remove_duplicates(df, subset=["title", "text"])
df["combined_text"] = df.apply(
    lambda row: combine_title_text(row.get("title", ""), row.get("text", "")), axis=1
)
df["clean_text"] = df["combined_text"].apply(lambda x: clean_text(x, remove_source_markers=True))
df = df[df["clean_text"].str.strip() != ""].reset_index(drop=True)

X_train, X_test, y_train, y_test = prepare_train_test_split(
    df, text_column="clean_text", label_column="label",
    test_size=0.2, random_state=42
)
print(f"Test set: {len(X_test)} articles")

## 2. Load Trained Models

In [ ]:
model_dir = get_model_dir()
times_path = os.path.join("..", "reports", "results", "training_times.json")
train_times = {}
if os.path.exists(times_path):
    with open(times_path, "r", encoding="utf-8") as f:
        train_times = json.load(f)

model_names = ["DummyClassifier", "MultinomialNB", "LogisticRegression", "LinearSVC"]

model_results = {}
for name in model_names:
    safe = name.replace(" ", "_").lower()
    path = os.path.join(model_dir, f"{safe}.joblib")
    if os.path.exists(path):
        pipeline = joblib.load(path)
        t = train_times.get(name, {}).get("train_time_seconds", 0)
        model_results[name] = {"pipeline": pipeline, "train_time": t}
        print(f"Loaded {name} (train_time={t:.2f}s)")
    else:
        print(f"Not found: {name}")

## 3. Evaluate All Models

In [ ]:
all_metrics = evaluate_all_models(model_results, X_test, y_test)

## 4. Model Comparison Table

In [ ]:
comparison_df = create_comparison_table(all_metrics)
print("\n" + "="*60)
print("MODEL COMPARISON (sorted by F1-Score)")
print("="*60)
print(comparison_df.to_string(index=False))

## 5. Confusion Matrices

In [ ]:
for name, info in model_results.items():
    y_pred = info["pipeline"].predict(X_test)
    plot_confusion_matrix(y_test, y_pred, name)
    plt.show()

## 6. Visual Comparison

In [ ]:
plot_model_comparison(comparison_df)
plt.show()

## 7. Error Analysis — Best Model

In [ ]:
best_name = comparison_df.iloc[0]["Model"]
best_pipeline = model_results[best_name]["pipeline"]
print(f"Best model by F1-Score: {best_name}")
print(f"F1-Score: {comparison_df.iloc[0]['F1-Score']:.4f}")

error_analysis = analyze_errors(best_pipeline, X_test, y_test, best_name, n_examples=5)

### False Positive Examples (Real classified as Fake)

In [ ]:
print(f"Total False Positives: {error_analysis['false_positives_count']}")
for i, ex in enumerate(error_analysis["false_positive_examples"]):
    print(f"\n--- Example {i+1} ---")
    print(f"True: {ex['true_label']}  Predicted: {ex['predicted_label']}")
    print(f"Text: {ex['text_preview']}")

### False Negative Examples (Fake classified as Real)

In [ ]:
print(f"Total False Negatives: {error_analysis['false_negatives_count']}")
for i, ex in enumerate(error_analysis["false_negative_examples"]):
    print(f"\n--- Example {i+1} ---")
    print(f"True: {ex['true_label']}  Predicted: {ex['predicted_label']}")
    print(f"Text: {ex['text_preview']}")

## 8. Discussion

### Source Bias
The dataset contains articles from specific sources. Real articles often come from Reuters, while fake articles come from various unreliable websites.

### Temporal Bias
Articles cover a specific time period.

### Style vs. Content
The model may learn stylistic differences rather than factual accuracy.

### What the Model Actually Learns
The high performance should be interpreted cautiously. The model likely learns source-specific vocabulary, topic associations, and writing quality indicators.

## 9. Save Results

In [ ]:
save_all_metrics(all_metrics)

from src.train import save_pipeline as sp
sp(best_pipeline, "best_model.joblib")

from src.utils import save_results_json
save_results_json(error_analysis, "error_analysis.json")
print("All evaluation results saved.")

## 10. Conclusions

1. **Best model**: **LinearSVC** achieved the highest performance (Accuracy = 0.9968, Precision = 0.9955, Recall = 0.9986, F1-Score = 0.9971), with only 25 errors out of 7,820 test articles.
2. **Most problematic error type**: Fake articles classified as Real (19 cases) outnumber Real articles classified as Fake (6 cases); those 19 cases typically mimic formal journalistic style on real events.
3. **Does the model generalize?**: Within the 2015–2017 ISOT corpus, linear TF-IDF models generalize very strongly, though out-of-domain or very short articles (e.g., post-2017 news) can still trigger errors because the classifier learns stylistic and vocabulary patterns rather than fact-checking.